# TurboLLM on Kaggle — dual T4 (auto-tune vs chat)

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, and Internet -> **On**.

Then run the cells top to bottom. The first run builds a CUDA engine (~30-40 min, one
time — it is cached in `/kaggle/working`); later runs skip straight to serving.

Everything runs from the branch, so the loop is: fix locally -> push -> re-run cell 1
(`git pull`) -> re-run the serve/test cells.

In [ ]:
# 1) Get the branch (clone the first time, fast-forward after)
!git clone -b claude/turbollm-runpod-dual-gpu-7aa8c0 https://github.com/mohitsoni48/TurboLLM.git 2>/dev/null; cd /kaggle/working/TurboLLM && git pull

In [ ]:
# 2) One-time heavy setup: Node, web build, native CUDA build of the TurboQuant fork,
#    and the Q4 model download. Idempotent - re-runs skip anything already done.
!bash /kaggle/working/TurboLLM/deploy/kaggle/setup.sh

In [ ]:
# 3) Start the daemon from source + register/activate the CUDA engine + open the public
#    GUI tunnel. Prints the *.trycloudflare.com URL and the Token needed to open it.
!bash /kaggle/working/TurboLLM/deploy/kaggle/serve.sh start

In [ ]:
# 4) THE TEST: auto-tune -> save winner -> load -> measure real chat tok/s -> sample both
#    T4s -> diff winner-vs-loaded config. Prints auto-tune tps vs chat tps side by side.
!cd /kaggle/working/TurboLLM && python3 deploy/kaggle/bench_vs_chat.py --ctx 8192

In [ ]:
# 5) Manual dual-GPU checks (run during a chat to see both GPUs busy)
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!echo '--- sysinfo (expect 2 GPUs) ---'; curl -s localhost:6996/api/v1/sysinfo | python3 -m json.tool
!echo '--- active engine ---'; curl -s localhost:6996/api/v1/engines | python3 -m json.tool

## Dev loop

- **daemon/src change:** re-run cell 1 (`git pull`), then
  `!cd /kaggle/working/TurboLLM && bash deploy/kaggle/serve.sh restart`
- **web UI change:** re-run cell 1, then
  `!cd /kaggle/working/TurboLLM/turbollm && npm run build:web && bash ../deploy/kaggle/serve.sh restart`

Knobs: `TURBOLLM_MODEL_FILE` (default `Qwen3.6-27B-Q4_K_M.gguf`), `TURBOLLM_BUILD_JOBS`
(compile parallelism), `--model KEY` / `--ctx N` on the test script.